### Import libraries

In [1]:
import pandas as pd
from pathlib import Path

from sqlalchemy import create_engine, text

### Define warehouse path

In [2]:
WAREHOUSE_DATA_PATH = Path(
    "../data/processed/warehouse"
)

print(
    "Warehouse folder exists:",
    WAREHOUSE_DATA_PATH.exists()
)

Warehouse folder exists: True


### Create the SQL Server connection

In [3]:
server = r"localhost\SQLEXPRESS"
database = "rivercare_analytics"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

engine = create_engine(
    connection_string,
    fast_executemany=True
)

### Test the connection

In [4]:
with engine.connect() as connection:

    result = connection.execute(
        text(
            """
            SELECT
                @@SERVERNAME AS ServerName,
                DB_NAME() AS DatabaseName
            """
        )
    )

    for row in result:
        print(row)

('LAPTOP-5P4TH31J\\SQLEXPRESS', 'rivercare_analytics')


In [5]:
from sqlalchemy import inspect, text
import pandas as pd


def refresh_existing_fact_table(
    table_name,
    key_column
):
    # Load newly rebuilt warehouse CSV
    df = pd.read_csv(
        WAREHOUSE_DATA_PATH / f"{table_name}.csv"
    )

    temp_table = f"_refresh_{table_name}"

    print(
        f"Loading temporary table for {table_name}..."
    )

    # Temporary copy of corrected warehouse data
    df.to_sql(
        temp_table,
        engine,
        schema="dbo",
        if_exists="replace",
        index=False,
        chunksize=1000
    )

    # Get actual SQL Server columns
    inspector = inspect(engine)

    sql_columns = [
        col["name"]
        for col in inspector.get_columns(
            table_name,
            schema="dbo"
        )
    ]

    # Update every matching column except the primary key
    update_columns = [
        col
        for col in df.columns
        if col in sql_columns
        and col != key_column
    ]

    set_clause = ",\n".join(
        [
            f"t.[{col}] = s.[{col}]"
            for col in update_columns
        ]
    )

    update_sql = f"""
    UPDATE t

    SET
        {set_clause}

    FROM dbo.[{table_name}] AS t

    INNER JOIN dbo.[{temp_table}] AS s
        ON t.[{key_column}]
         = s.[{key_column}];
    """

    with engine.begin() as conn:

        result = conn.execute(
            text(update_sql)
        )

        conn.execute(
            text(
                f"DROP TABLE dbo.[{temp_table}]"
            )
        )

    print(
        f"{table_name}: {result.rowcount} rows refreshed."
    )


refresh_existing_fact_table(
    "fact_encounter",
    "EncounterKey"
)

refresh_existing_fact_table(
    "fact_admission",
    "AdmissionKey"
)

Loading temporary table for fact_encounter...
fact_encounter: 90000 rows refreshed.
Loading temporary table for fact_admission...
fact_admission: 17740 rows refreshed.


### Confirm Python can see 13 CSVs

In [5]:
expected_files = [
    "dim_patient.csv",
    "dim_department.csv",
    "dim_provider.csv",
    "dim_payer.csv",
    "dim_diagnosis.csv",
    "dim_procedure.csv",
    "dim_date.csv",

    "fact_encounter.csv",
    "fact_admission.csv",
    "fact_appointment.csv",
    "fact_encounter_diagnosis.csv",
    "fact_encounter_procedure.csv",
    "fact_lab_result.csv"
]

for filename in expected_files:

    path = WAREHOUSE_DATA_PATH / filename

    print(
        filename,
        "✅ EXISTS"
        if path.exists()
        else "❌ MISSING"
    )

dim_patient.csv ✅ EXISTS
dim_department.csv ✅ EXISTS
dim_provider.csv ✅ EXISTS
dim_payer.csv ✅ EXISTS
dim_diagnosis.csv ✅ EXISTS
dim_procedure.csv ✅ EXISTS
dim_date.csv ✅ EXISTS
fact_encounter.csv ✅ EXISTS
fact_admission.csv ✅ EXISTS
fact_appointment.csv ✅ EXISTS
fact_encounter_diagnosis.csv ✅ EXISTS
fact_encounter_procedure.csv ✅ EXISTS
fact_lab_result.csv ✅ EXISTS


In [6]:
available_files = sum(
    (WAREHOUSE_DATA_PATH / filename).exists()
    for filename in expected_files
)

print(
    f"Warehouse CSV files available: "
    f"{available_files}/13"
)

Warehouse CSV files available: 13/13


### Check CSV columns against SQL Server columns

In [7]:
from sqlalchemy import inspect

inspector = inspect(engine)

table_names = [
    "dim_patient",
    "dim_department",
    "dim_provider",
    "dim_payer",
    "dim_diagnosis",
    "dim_procedure",
    "dim_date",

    "fact_encounter",
    "fact_admission",
    "fact_appointment",
    "fact_encounter_diagnosis",
    "fact_encounter_procedure",
    "fact_lab_result"
]

In [8]:
schema_check_results = []

for table_name in table_names:

    csv_path = (
        WAREHOUSE_DATA_PATH
        / f"{table_name}.csv"
    )

    # Read only column headers
    csv_df = pd.read_csv(
        csv_path,
        nrows=0
    )

    csv_columns = list(
        csv_df.columns
    )

    sql_columns = [
        column["name"]
        for column
        in inspector.get_columns(
            table_name,
            schema="dbo"
        )
    ]

    missing_in_sql = [
        column
        for column in csv_columns
        if column not in sql_columns
    ]

    missing_in_csv = [
        column
        for column in sql_columns
        if column not in csv_columns
    ]

    schema_check_results.append({
        "Table": table_name,
        "CSVColumns": len(csv_columns),
        "SQLColumns": len(sql_columns),
        "MissingInSQL": missing_in_sql,
        "MissingInCSV": missing_in_csv,
        "Match": (
            len(missing_in_sql) == 0
            and
            len(missing_in_csv) == 0
        )
    })

schema_check_df = pd.DataFrame(
    schema_check_results
)

schema_check_df[
    [
        "Table",
        "CSVColumns",
        "SQLColumns",
        "Match"
    ]
]

,Table,CSVColumns,SQLColumns,Match
0,dim_patient,11,11,True
1,dim_department,7,7,True
2,dim_provider,8,8,True
3,dim_payer,4,4,True
4,dim_diagnosis,6,6,True
5,dim_procedure,6,6,True
6,dim_date,12,12,True
7,fact_encounter,26,26,True
8,fact_admission,19,19,True
9,fact_appointment,18,21,False


In [9]:
for table_name in [
    "fact_appointment",
    "fact_lab_result"
]:

    row = schema_check_df[
        schema_check_df["Table"] == table_name
    ].iloc[0]

    print("\nTABLE:", table_name)

    print(
        "CSV columns missing in SQL:",
        row["MissingInSQL"]
    )

    print(
        "SQL columns missing in CSV:",
        row["MissingInCSV"]
    )


TABLE: fact_appointment
CSV columns missing in SQL: []
SQL columns missing in CSV: ['StatusKPIEligibleFlag', 'BookingLeadKPIEligibleFlag', 'AppointmentQualityFlag']

TABLE: fact_lab_result
CSV columns missing in SQL: []
SQL columns missing in CSV: ['CleanValueKPIEligibleFlag', 'LabQualityFlag']


### Rerun the Python schema check

In [10]:
from sqlalchemy import inspect

inspector = inspect(engine)

In [11]:
schema_check_results = []

for table_name in table_names:

    csv_path = (
        WAREHOUSE_DATA_PATH
        / f"{table_name}.csv"
    )

    csv_df = pd.read_csv(
        csv_path,
        nrows=0
    )

    csv_columns = list(
        csv_df.columns
    )

    sql_columns = [
        column["name"]
        for column
        in inspector.get_columns(
            table_name,
            schema="dbo"
        )
    ]

    missing_in_sql = [
        column
        for column in csv_columns
        if column not in sql_columns
    ]

    missing_in_csv = [
        column
        for column in sql_columns
        if column not in csv_columns
    ]

    schema_check_results.append({
        "Table": table_name,
        "CSVColumns": len(csv_columns),
        "SQLColumns": len(sql_columns),
        "MissingInSQL": missing_in_sql,
        "MissingInCSV": missing_in_csv,
        "Match": (
            len(missing_in_sql) == 0
            and
            len(missing_in_csv) == 0
        )
    })

schema_check_df = pd.DataFrame(
    schema_check_results
)

schema_check_df[
    [
        "Table",
        "CSVColumns",
        "SQLColumns",
        "Match"
    ]
]

,Table,CSVColumns,SQLColumns,Match
0,dim_patient,11,11,True
1,dim_department,7,7,True
2,dim_provider,8,8,True
3,dim_payer,4,4,True
4,dim_diagnosis,6,6,True
5,dim_procedure,6,6,True
6,dim_date,12,12,True
7,fact_encounter,26,26,True
8,fact_admission,19,19,True
9,fact_appointment,18,18,True


In [12]:
print(
    "Tables with schema mismatch:",
    (
        schema_check_df["Match"] == False
    ).sum()
)

Tables with schema mismatch: 0


### Confirm SQL tables are empty

In [13]:
dimension_tables = [
    "dim_patient",
    "dim_department",
    "dim_provider",
    "dim_payer",
    "dim_diagnosis",
    "dim_procedure",
    "dim_date"
]

with engine.connect() as connection:

    for table_name in dimension_tables:

        count = connection.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM dbo.{table_name}
                """
            )
        ).scalar()

        print(
            f"{table_name}: {count:,} rows"
        )

dim_patient: 0 rows
dim_department: 0 rows
dim_provider: 0 rows
dim_payer: 0 rows
dim_diagnosis: 0 rows
dim_procedure: 0 rows
dim_date: 0 rows


### Load all 7 dimensions

In [14]:
dimension_tables = [
    "dim_patient",
    "dim_department",
    "dim_provider",
    "dim_payer",
    "dim_diagnosis",
    "dim_procedure",
    "dim_date"
]

for table_name in dimension_tables:

    csv_path = (
        WAREHOUSE_DATA_PATH
        / f"{table_name}.csv"
    )

    df = pd.read_csv(csv_path)

    print(
        f"Loading {table_name}: "
        f"{len(df):,} rows..."
    )

    df.to_sql(
        name=table_name,
        con=engine,
        schema="dbo",
        if_exists="append",
        index=False,
        chunksize=1000
    )

    print(
        f"✅ {table_name} loaded"
    )

Loading dim_patient: 9,981 rows...
✅ dim_patient loaded
Loading dim_department: 26 rows...
✅ dim_department loaded
Loading dim_provider: 121 rows...
✅ dim_provider loaded
Loading dim_payer: 7 rows...
✅ dim_payer loaded
Loading dim_diagnosis: 51 rows...
✅ dim_diagnosis loaded
Loading dim_procedure: 41 rows...
✅ dim_procedure loaded
Loading dim_date: 1,336 rows...
✅ dim_date loaded


### Validate dimension row counts

In [15]:
dimension_validation = []

with engine.connect() as connection:

    for table_name in dimension_tables:

        csv_df = pd.read_csv(
            WAREHOUSE_DATA_PATH
            / f"{table_name}.csv"
        )

        csv_rows = len(csv_df)

        sql_rows = connection.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM dbo.{table_name}
                """
            )
        ).scalar()

        dimension_validation.append({
            "Table": table_name,
            "CSVRows": csv_rows,
            "SQLRows": sql_rows,
            "Difference": sql_rows - csv_rows,
            "Match": sql_rows == csv_rows
        })

dimension_validation_df = pd.DataFrame(
    dimension_validation
)

dimension_validation_df

,Table,CSVRows,SQLRows,Difference,Match
0,dim_patient,9981,9981,0,True
1,dim_department,26,26,0,True
2,dim_provider,121,121,0,True
3,dim_payer,7,7,0,True
4,dim_diagnosis,51,51,0,True
5,dim_procedure,41,41,0,True
6,dim_date,1336,1336,0,True


In [16]:
print(
    "Dimension tables with row-count mismatch:",
    (
        dimension_validation_df[
            "Match"
        ] == False
    ).sum()
)

Dimension tables with row-count mismatch: 0


### Confirm UNKNOWN members survived the SQL load

In [17]:
unknown_queries = {
    "dim_patient":
        "PatientKey = 0 AND PatientID = 'UNKNOWN'",

    "dim_department":
        "DepartmentKey = 0 AND DepartmentID = 'UNKNOWN'",

    "dim_provider":
        "ProviderKey = 0 AND ProviderID = 'UNKNOWN'",

    "dim_payer":
        "PayerKey = 0 AND PayerID = 'UNKNOWN'",

    "dim_diagnosis":
        "DiagnosisKey = 0 AND DiagnosisID = 'UNKNOWN'",

    "dim_procedure":
        "ProcedureKey = 0 AND ProcedureID = 'UNKNOWN'",

    "dim_date":
        "DateKey = 0"
}

with engine.connect() as connection:

    for table_name, condition in unknown_queries.items():

        count = connection.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM dbo.{table_name}
                WHERE {condition}
                """
            )
        ).scalar()

        print(
            f"{table_name} UNKNOWN member: {count}"
        )

dim_patient UNKNOWN member: 1
dim_department UNKNOWN member: 1
dim_provider UNKNOWN member: 1
dim_payer UNKNOWN member: 1
dim_diagnosis UNKNOWN member: 1
dim_procedure UNKNOWN member: 1
dim_date UNKNOWN member: 1


### Load fact_encounter first
### Confirm fact_encounter is currently empty

In [18]:
with engine.connect() as connection:

    count = connection.execute(
        text(
            """
            SELECT COUNT(*)
            FROM dbo.fact_encounter
            """
        )
    ).scalar()

    print(
        "fact_encounter SQL rows:",
        count
    )

fact_encounter SQL rows: 0


### Load the CSV

In [19]:
fact_encounter_load = pd.read_csv(
    WAREHOUSE_DATA_PATH
    / "fact_encounter.csv"
)

print(
    "CSV rows:",
    f"{len(fact_encounter_load):,}"
)

print(
    "Columns:",
    len(fact_encounter_load.columns)
)

CSV rows: 90,000
Columns: 26


In [20]:
print(
    f"Loading fact_encounter: "
    f"{len(fact_encounter_load):,} rows..."
)

fact_encounter_load.to_sql(
    name="fact_encounter",
    con=engine,
    schema="dbo",
    if_exists="append",
    index=False,
    chunksize=1000
)

print(
    "✅ fact_encounter loaded successfully"
)

Loading fact_encounter: 90,000 rows...
✅ fact_encounter loaded successfully


### Validate CSV vs SQL row count

In [21]:
with engine.connect() as connection:

    sql_rows = connection.execute(
        text(
            """
            SELECT COUNT(*)
            FROM dbo.fact_encounter
            """
        )
    ).scalar()

csv_rows = len(
    fact_encounter_load
)

print(
    "CSV rows:",
    f"{csv_rows:,}"
)

print(
    "SQL rows:",
    f"{sql_rows:,}"
)

print(
    "Difference:",
    sql_rows - csv_rows
)

print(
    "Match:",
    sql_rows == csv_rows
)

CSV rows: 90,000
SQL rows: 90,000
Difference: 0
Match: True


### Validate the primary key in SQL

In [22]:
with engine.connect() as connection:

    result = connection.execute(
        text(
            """
            SELECT
                COUNT(*) AS TotalRows,
                COUNT(DISTINCT EncounterKey)
                    AS UniqueEncounterKeys,
                COUNT(DISTINCT EncounterID)
                    AS UniqueEncounterIDs
            FROM dbo.fact_encounter
            """
        )
    ).fetchone()

print(
    "Total rows:",
    result[0]
)

print(
    "Unique EncounterKeys:",
    result[1]
)

print(
    "Unique EncounterIDs:",
    result[2]
)

Total rows: 90000
Unique EncounterKeys: 90000
Unique EncounterIDs: 90000


### Test the foreign-key relationships

In [23]:
with engine.connect() as connection:

    orphan_check = connection.execute(
        text(
            """
            SELECT

                SUM(
                    CASE
                        WHEN p.PatientKey IS NULL
                        THEN 1 ELSE 0
                    END
                ) AS InvalidPatient,

                SUM(
                    CASE
                        WHEN pr.ProviderKey IS NULL
                        THEN 1 ELSE 0
                    END
                ) AS InvalidProvider,

                SUM(
                    CASE
                        WHEN d.DepartmentKey IS NULL
                        THEN 1 ELSE 0
                    END
                ) AS InvalidDepartment,

                SUM(
                    CASE
                        WHEN py.PayerKey IS NULL
                        THEN 1 ELSE 0
                    END
                ) AS InvalidPayer,

                SUM(
                    CASE
                        WHEN dt.DateKey IS NULL
                        THEN 1 ELSE 0
                    END
                ) AS InvalidDate

            FROM dbo.fact_encounter f

            LEFT JOIN dbo.dim_patient p
                ON f.PatientKey = p.PatientKey

            LEFT JOIN dbo.dim_provider pr
                ON f.ProviderKey = pr.ProviderKey

            LEFT JOIN dbo.dim_department d
                ON f.DepartmentKey = d.DepartmentKey

            LEFT JOIN dbo.dim_payer py
                ON f.PayerKey = py.PayerKey

            LEFT JOIN dbo.dim_date dt
                ON f.EncounterDateKey = dt.DateKey;
            """
        )
    ).fetchone()

print(
    "Invalid Patient:",
    orphan_check[0]
)

print(
    "Invalid Provider:",
    orphan_check[1]
)

print(
    "Invalid Department:",
    orphan_check[2]
)

print(
    "Invalid Payer:",
    orphan_check[3]
)

print(
    "Invalid Date:",
    orphan_check[4]
)

Invalid Patient: 0
Invalid Provider: 0
Invalid Department: 0
Invalid Payer: 0
Invalid Date: 0


### Check UNKNOWN mappings in SQL

In [24]:
with engine.connect() as connection:

    unknowns = connection.execute(
        text(
            """
            SELECT
                SUM(
                    CASE WHEN PatientKey = 0
                    THEN 1 ELSE 0 END
                ) AS UnknownPatient,

                SUM(
                    CASE WHEN ProviderKey = 0
                    THEN 1 ELSE 0 END
                ) AS UnknownProvider,

                SUM(
                    CASE WHEN DepartmentKey = 0
                    THEN 1 ELSE 0 END
                ) AS UnknownDepartment,

                SUM(
                    CASE WHEN PayerKey = 0
                    THEN 1 ELSE 0 END
                ) AS UnknownPayer,

                SUM(
                    CASE WHEN EncounterDateKey = 0
                    THEN 1 ELSE 0 END
                ) AS UnknownDate

            FROM dbo.fact_encounter;
            """
        )
    ).fetchone()

print(
    "UNKNOWN Patient:",
    unknowns[0]
)

print(
    "UNKNOWN Provider:",
    unknowns[1]
)

print(
    "UNKNOWN Department:",
    unknowns[2]
)

print(
    "UNKNOWN Payer:",
    unknowns[3]
)

print(
    "UNKNOWN Date:",
    unknowns[4]
)

UNKNOWN Patient: 176
UNKNOWN Provider: 1401
UNKNOWN Department: 216
UNKNOWN Payer: 0
UNKNOWN Date: 0


### First make sure fact tables are empty

In [25]:
remaining_fact_tables = [
    "fact_admission",
    "fact_appointment",
    "fact_encounter_diagnosis",
    "fact_encounter_procedure",
    "fact_lab_result"
]

with engine.connect() as connection:

    for table_name in remaining_fact_tables:

        count = connection.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM dbo.{table_name}
                """
            )
        ).scalar()

        print(
            f"{table_name}: {count:,} rows"
        )

fact_admission: 0 rows
fact_appointment: 0 rows
fact_encounter_diagnosis: 0 rows
fact_encounter_procedure: 0 rows
fact_lab_result: 0 rows


### Load the 5 facts

In [26]:
for table_name in remaining_fact_tables:

    csv_path = (
        WAREHOUSE_DATA_PATH
        / f"{table_name}.csv"
    )

    df = pd.read_csv(csv_path)

    print(
        f"\nLoading {table_name}: "
        f"{len(df):,} rows..."
    )

    df.to_sql(
        name=table_name,
        con=engine,
        schema="dbo",
        if_exists="append",
        index=False,
        chunksize=1000
    )

    print(
        f"✅ {table_name} loaded successfully"
    )


Loading fact_admission: 17,740 rows...
✅ fact_admission loaded successfully

Loading fact_appointment: 50,000 rows...
✅ fact_appointment loaded successfully

Loading fact_encounter_diagnosis: 153,137 rows...
✅ fact_encounter_diagnosis loaded successfully

Loading fact_encounter_procedure: 70,419 rows...
✅ fact_encounter_procedure loaded successfully

Loading fact_lab_result: 100,000 rows...
✅ fact_lab_result loaded successfully


### Validate all 6 fact row counts

In [27]:
all_fact_tables = [
    "fact_encounter",
    "fact_admission",
    "fact_appointment",
    "fact_encounter_diagnosis",
    "fact_encounter_procedure",
    "fact_lab_result"
]

fact_validation = []

with engine.connect() as connection:

    for table_name in all_fact_tables:

        csv_df = pd.read_csv(
            WAREHOUSE_DATA_PATH
            / f"{table_name}.csv"
        )

        csv_rows = len(csv_df)

        sql_rows = connection.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM dbo.{table_name}
                """
            )
        ).scalar()

        fact_validation.append({
            "Table": table_name,
            "CSVRows": csv_rows,
            "SQLRows": sql_rows,
            "Difference": sql_rows - csv_rows,
            "Match": sql_rows == csv_rows
        })

fact_validation_df = pd.DataFrame(
    fact_validation
)

fact_validation_df

,Table,CSVRows,SQLRows,Difference,Match
0,fact_encounter,90000,90000,0,True
1,fact_admission,17740,17740,0,True
2,fact_appointment,50000,50000,0,True
3,fact_encounter_diagnosis,153137,153137,0,True
4,fact_encounter_procedure,70419,70419,0,True
5,fact_lab_result,100000,100000,0,True


In [28]:
print(
    "Fact tables with row-count mismatch:",
    (
        fact_validation_df["Match"] == False
    ).sum()
)

Fact tables with row-count mismatch: 0


### Validate all 13 SQL tables together

In [29]:
all_sql_tables = [
    "dim_patient",
    "dim_department",
    "dim_provider",
    "dim_payer",
    "dim_diagnosis",
    "dim_procedure",
    "dim_date",

    "fact_encounter",
    "fact_admission",
    "fact_appointment",
    "fact_encounter_diagnosis",
    "fact_encounter_procedure",
    "fact_lab_result"
]

sql_inventory = []

with engine.connect() as connection:

    for table_name in all_sql_tables:

        row_count = connection.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM dbo.{table_name}
                """
            )
        ).scalar()

        sql_inventory.append({
            "Table": table_name,
            "SQLRows": row_count
        })

sql_inventory_df = pd.DataFrame(
    sql_inventory
)

sql_inventory_df

,Table,SQLRows
0,dim_patient,9981
1,dim_department,26
2,dim_provider,121
3,dim_payer,7
4,dim_diagnosis,51
5,dim_procedure,41
6,dim_date,1336
7,fact_encounter,90000
8,fact_admission,17740
9,fact_appointment,50000


In [6]:
sql_validation_query = """
SELECT
    COUNT(*) AS TotalAdmissions,

    SUM(
        CASE
            WHEN LengthOfStayDays > 10
            THEN 1
            ELSE 0
        END
    ) AS AdmissionsLOSAbove10,

    SUM(
        CASE
            WHEN LengthOfStayDays < 0
            THEN 1
            ELSE 0
        END
    ) AS NegativeLOS,

    MIN(LengthOfStayDays) AS MinimumLOS,

    MAX(LengthOfStayDays) AS MaximumLOS

FROM dbo.fact_admission;
"""

sql_los_validation = pd.read_sql(
    sql_validation_query,
    engine
)

sql_los_validation

,TotalAdmissions,AdmissionsLOSAbove10,NegativeLOS,MinimumLOS,MaximumLOS
0,17740,0,0,0.0202,7.1219


In [7]:
encounter_validation_query = """
SELECT
    COUNT(*) AS TotalEncounters,

    SUM(
        CASE
            WHEN EncounterDurationHours < 0
            THEN 1
            ELSE 0
        END
    ) AS NegativeEncounterDurations,

    MIN(EncounterDurationHours) AS MinimumEncounterDuration,

    MAX(EncounterDurationHours) AS MaximumEncounterDuration,

    SUM(
        CASE
            WHEN EncounterEndDateTime IS NULL
            THEN 1
            ELSE 0
        END
    ) AS MissingEncounterEnds

FROM dbo.fact_encounter;
"""

sql_encounter_validation = pd.read_sql(
    encounter_validation_query,
    engine
)

sql_encounter_validation

,TotalEncounters,NegativeEncounterDurations,MinimumEncounterDuration,MaximumEncounterDuration,MissingEncounterEnds
0,90000,0,0.0,59.9983,0
